# 第15回　ニューラルネットワーク実装（全結合1層）
***
> **前提**: 第14回 PyTorch 基礎の続きです。第7回では `MyLinearRegression` クラスを自作し `fit` / `predict` を実装しました。PyTorch では `nn.Module` を継承し `forward` メソッドで同様の「入力→出力」の計算を定義します。
>
> | 第7回 `MyLinearRegression` | 第15回 `nn.Module` |
> |---|---|
> | `fit(X, y)` で重みを学習 | 学習ループ + `optimizer.step()` で重みを更新 |
> | `predict(X)` で予測 | `forward(x)` で順伝播 |
> | `score(X, y)` で R² | テストデータで正解率を計算 |

## 目次
1. SimpleMLP の実装
2. 学習ループ
3. 損失曲線
4. テスト正解率

---

## この回で学ぶこと

### ニューラルネットワークの構造

多層パーセプトロン（MLP：Multilayer Perceptron）は，複数の「全結合層（Linear層）」を重ねたシンプルなニューラルネットワークだ：

```
入力層 (784次元)
   ↓ nn.Linear(784, 128) 重み行列 W₁ (784×128) + バイアス b₁
隠れ層 (128次元)
   ↓ ReLU(x) = max(0, x)  活性化関数
   ↓ nn.Linear(128, 10)  重み行列 W₂ (128×10) + バイアス b₂
出力層 (10次元) ← 数字0〜9のスコア（logits）
```

全部で 784×128 + 128 + 128×10 + 10 = **101,770個のパラメータ**が学習される。

### なぜ活性化関数（ReLU）が必要か

線形層だけを重ねると，どれだけ重ねても「1つの線形変換」と等価になってしまう（行列の積は行列）。**非線形な活性化関数**を挟むことで，複雑な非線形パターンを学習できるようになる。

ReLU（Rectified Linear Unit）: `f(x) = max(0, x)`
- 計算が単純で速い
- **勾配消失問題を回避**（sigmoid や tanh は深いネットで勾配が消えやすかった）
- 現在のデフォルト選択

### 誤差逆伝播法（Backpropagation）の流れ

```
① forward(x)  : 入力から出力を計算（順伝播）
② loss = criterion(output, label)  : 損失を計算
③ optimizer.zero_grad()  : 前のステップの勾配をリセット
④ loss.backward()  : 各パラメータの勾配を計算（逆伝播）
⑤ optimizer.step()  : 勾配を使ってパラメータを更新
```

**③ `zero_grad()` を忘れると**：勾配が前のステップと加算され，誤った更新が起きる。必ずループの最初でリセットすること。

### Adam オプティマイザとは

勾配降下法（SGD）の改良版で，以下の特徴を持つ：
- **各パラメータに個別の学習率**を自動調整
- モーメンタム（過去の勾配の方向を考慮）
- `lr=0.001` が一般的なデフォルト設定

SGD より収束が速く，初心者にも扱いやすいため，現在の標準的な選択肢となっている。

In [ ]:
%pip install -q torch torchvision


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## 問題1　SimpleMLP の実装
***

### `nn.Module` の仕組み

PyTorch のモデルは `nn.Module` を継承したクラスとして実装する。覚えるべきルールは2つ：

1. `__init__` で使用する層を**定義**する（ここでパラメータが初期化される）
2. `forward` で入力から出力への**計算を記述**する

```python
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()  # ← 必須！親クラスの初期化
        # 層を定義
        self.fc1 = nn.Linear(784, 128)  # 重み行列 (784, 128) を作成
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # 入力 x: (batch_size, 1, 28, 28)
        x = x.view(-1, 784)       # → (batch_size, 784) に flatten
        x = self.relu(self.fc1(x))  # → 線形変換 → ReLU → (batch_size, 128)
        return self.fc2(x)        # → (batch_size, 10) logits を返す
```

### 出力層に softmax/sigmoid がない理由

`CrossEntropyLoss`（問題2で使用）は内部で softmax を計算するため，`forward` の出力は「確率」ではなく「生スコア（logits）」でよい。推論時（`predict_proba`に相当）には `torch.softmax(output, dim=1)` を別途適用する。

### 課題

以下の要件を満たす MLP クラス `SimpleMLP(nn.Module)` を実装してください。

- 入力: 784（28×28 flatten）
- 隠れ層: 128 ユニット + ReLU
- 出力: 10 クラス（logits）

実装後，`model = SimpleMLP()` と `print(model)` でモデル構造を確認してください。

#### Hints
- `nn.Module` を継承したクラスを作る。`super().__init__()` の呼び出しは必須
- `__init__` では使用する層（`nn.Linear`, `nn.ReLU` など）をインスタンス変数として定義する
- `forward` の最初で `x.view(-1, 784)` を使って画像を1次元ベクトルに変換する（`-1` はバッチサイズを自動計算）
- 出力層（10クラス分）には活性化関数を付けない（`CrossEntropyLoss` が内部で処理するため）
- 実装後 `print(model)` でアーキテクチャを確認できる

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # ここにあなたのコードを書いてください

    def forward(self, x):
        # ここにあなたのコードを書いてください
        pass


## 問題2　学習ループの実装
***

### 学習ループの各ステップ解説

```python
for epoch in range(3):          # ← 全データを3周する
    model.train()               # ← 訓練モードに切り替え（Dropout等が有効になる）
    for images, labels in train_loader:   # ← ミニバッチを順に取り出す
        images = images.to(device)        # ← データをGPU/CPUに送る
        labels = labels.to(device)

        optimizer.zero_grad()             # ← ① 勾配をリセット（必須！）
        outputs = model(images)           # ← ② forward: 予測を計算
        loss = criterion(outputs, labels) # ← ③ 損失を計算
        loss.backward()                   # ← ④ backward: 勾配を計算
        optimizer.step()                  # ← ⑤ パラメータを更新
```

### CrossEntropyLoss の意味

多クラス分類の標準的な損失関数だ：
```
CrossEntropyLoss = -Σ y_true × log(softmax(y_pred))
```
- 正解クラスの予測確率が高いほど損失が小さい（0に近づく）
- 損失が下がり続ければ，学習が正しく進んでいることを示す

### `loss.item()` の必要性

`loss` は Tensor であり，計算グラフを保持している。`.item()` を使うと Python の float に変換され，メモリが解放される。ループ内で `.item()` せずに損失を蓄積すると，計算グラフが蓄積されてメモリがあふれる危険がある。

### 課題

`SimpleMLP` を MNIST で学習してください（エポック数: 3, 損失関数: `CrossEntropyLoss`, 最適化: `Adam(lr=0.001)`）。

各エポックの平均訓練損失を出力してください。損失が徐々に下がっていれば学習が正しく進んでいる。

#### Hints
- モデル・データは同じデバイスに置く必要がある。`.to(device)` を忘れずに
- 1バッチの学習は必ず ① `zero_grad()` → ② モデルに入力して loss 計算 → ③ `loss.backward()` → ④ `optimizer.step()` の順番で行う
- `zero_grad()` を忘れると勾配が蓄積されてしまうため、ループの先頭で必ず呼ぶ
- `loss.item()` で Tensor から Python の数値に変換してから蓄積する（`.item()` なしで蓄積するとメモリが解放されない）
- エポックごとの平均損失は `total_loss / len(train_loader)` で計算できる

In [ ]:
# 学習ループ
# ここにあなたのコードを書いてください


## 問題3　損失曲線の記録と描画
***

### 損失曲線（Loss Curve）から何がわかるか

損失曲線はニューラルネットワークの学習状態を診断するための基本ツールだ：

```
【正常な学習】
損失: 2.5 → 1.2 → 0.8 → 0.5 → ...（単調に減少）

【学習率が高すぎる】
損失: 乱高下する，または発散（NaN になる）

【学習率が低すぎる】
損失: ほとんど変化しない

【収束後（過学習が始まると）】
訓練損失: 下がり続ける
検証損失: 下がった後に増加 → これを見たら早期終了（Early Stopping）
```

### 第13回との対応

第13回では決定木の「学習曲線（スコア vs データサイズ）」を描いた。今回は「損失曲線（損失 vs エポック数）」を描く。どちらも「学習の進行状況を可視化する」という目的は同じだ。

### 損失をリストに記録する方法

問題2の学習ループに `epoch_losses` リストを追加して記録する：

```python
epoch_losses = []
for epoch in range(3):
    # ... 学習ループ ...
    avg_loss = total_loss / len(train_loader)
    epoch_losses.append(avg_loss)
```

### 課題

問題2の学習を再実装し，各エポックの平均損失を `epoch_losses` リストに記録してください。

損失曲線（横軸: エポック数, 縦軸: 損失）を描画してください。グリッド線（`plt.grid(True)`）を加えると見やすくなる。


In [ ]:
# 損失曲線
# ここにあなたのコードを書いてください


## 問題4　テスト正解率の計算
***

### `model.eval()` と `torch.no_grad()` の意味

推論（テスト）時には，学習時と異なる2点の設定が必要だ：

**① `model.eval()`**：
- Dropout 層を無効化（学習時はランダムにニューロンを無効化するが，推論時は全ニューロンを使う）
- BatchNorm 層の動作を変更（学習時の統計量ではなく，学習済み統計量を使う）
- これを忘れると，推論のたびに結果が変わる（Dropoutのランダム性が残るため）

**② `torch.no_grad()`**：
- 計算グラフを構築しない → **メモリ使用量が激減**
- 推論では勾配計算は不要なため，この最適化が効く
- `with torch.no_grad():` ブロック内では `backward()` が呼べない

### `argmax(dim=1)` の意味

モデルの出力 `outputs` は shape `(batch_size, 10)` の logits（各クラスのスコア）だ。各サンプルについて最も高いスコアのクラス番号を予測値とする：

```python
outputs = [[1.2, -0.5, 3.1, ...],  # スコアが最大は index=2 → 数字「2」と予測
            ...]
preds = outputs.argmax(dim=1)  # → [2, ...]
```

### 課題

学習済み `SimpleMLP` でテストデータの**正解率**を計算して出力してください。

> **目安**: MNIST でシンプルな MLP（1層，128ユニット，3エポック）では 97〜98% 程度の正解率が期待される。これは第16回の CNN と比較する基準になる。

#### Hints
- 推論前には必ず `model.eval()` を呼ぶ（Dropout などの挙動が変わる）
- `torch.no_grad()` ブロック内では勾配が計算されないためメモリ効率が良い。推論時は常に使う
- モデルの出力（logits）から予測クラスを得るには `argmax(dim=1)` を使う（`dim=1` はクラス方向）
- 正解数のカウント：`(予測 == 正解ラベル)` は True/False の Tensor なので `.sum()` で合計できる

In [ ]:
# テスト正解率
# ここにあなたのコードを書いてください
